# 04 - Kalibrasyon ve Eşik

Bu not defteri, model olasılıklarının güvenilirliğini ve karar eşiği etkisini değerlendirir.

İncelediğimiz başlıklar:
- Sigmoid / isotonic kalibrasyon karşılaştırması
- Youden J ve F2 eşikleri
- F1-optimum eşik
- 3 seviyeli risk kategorisi davranışı


In [ ]:
# Ortamı hazırlıyor ve proje kökünü Python yoluna ekliyoruz.
from pathlib import Path
import sys

proje_kok = Path.cwd().resolve()
while proje_kok.name != 'diyabet_risk_tahmini' and proje_kok.parent != proje_kok:
    proje_kok = proje_kok.parent

if proje_kok.name != 'diyabet_risk_tahmini':
    raise RuntimeError('Notebook proje kökünden veya alt klasörlerinden çalıştırılmalıdır.')

if str(proje_kok) not in sys.path:
    sys.path.insert(0, str(proje_kok))

print(f'Proje kökü: {proje_kok}')


In [ ]:
# Kalibrasyon ve eşik analizi için ihtiyaç duyduğumuz araçları içe aktarıyoruz.
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

from makine_ogrenmesi.kaynak.model_egitimi import model_pipeline_olustur
from makine_ogrenmesi.kaynak.kalibrasyon import kalibrasyon_karsilastir
from makine_ogrenmesi.kaynak.esik_analizi import (
    youden_j_esigi_hesapla,
    f2_esigi_hesapla,
    risk_esiklerini_olustur,
    risk_kategorisi_belirle,
)
from makine_ogrenmesi.kaynak.ozellik_yapilandirmasi import HEDEF_KOLONU, OZELLIK_KOLONLARI
from makine_ogrenmesi.kaynak.veri_yukleyici import veri_setini_yukle


In [ ]:
# Veriyi eğitim, kalibrasyon ve test olarak üç parçaya ayırıyoruz.
veri = veri_setini_yukle(proje_kok / 'makine_ogrenmesi' / 'veri' / 'ham' / 'diabetes.csv')
X = veri[OZELLIK_KOLONLARI]
y = veri[HEDEF_KOLONU]

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

X_train, X_cal, y_train, y_cal = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.25,
    random_state=42,
    stratify=y_train_full,
)

print('Eğitim:', X_train.shape, 'Kalibrasyon:', X_cal.shape, 'Test:', X_test.shape)


In [ ]:
# XGBoost pipeline'ını kuruyor ve rapordaki en iyi parametreleri varsa modele uyguluyoruz.
pipeline = model_pipeline_olustur()['xgboost']

rapor_yolu = proje_kok / 'makine_ogrenmesi' / 'raporlar' / 'degerlendirme' / 'model_degerlendirme_ozeti.json'
if rapor_yolu.exists():
    rapor = json.loads(rapor_yolu.read_text(encoding='utf-8'))
    en_iyi = rapor.get('en_iyi_model', {})
    if en_iyi.get('model_adi') == 'xgboost':
        params = en_iyi.get('en_iyi_parametreler', {})
        if params:
            pipeline.set_params(**params)
            print('Rapor parametreleri modele uygulandı.')

pipeline.fit(X_train, y_train)
print('Temel model eğitildi.')


In [ ]:
# Kalibrasyon öncesi ve sonrası Brier skorlarını karşılaştırıp en iyi yöntemi seçiyoruz.
kalibrasyon = kalibrasyon_karsilastir(
    model=pipeline,
    x_kalibrasyon_egitim=X_cal,
    y_kalibrasyon_egitim=y_cal,
    x_degerlendirme=X_test,
    y_degerlendirme=y_test,
    cv=3,
)

pd.Series(kalibrasyon['brier_skorlari'], name='brier').to_frame()


In [ ]:
# Youden J, F2 ve F1-optimum eşiklerini hesaplayıp karşılaştırma tablosu üretiyoruz.
en_iyi_kalibrator = kalibrasyon['en_iyi_kalibrator']
y_prob = en_iyi_kalibrator.predict_proba(X_test)[:, 1]

youden = youden_j_esigi_hesapla(y_test, y_prob)
f2 = f2_esigi_hesapla(y_test, y_prob, beta=2.0)
risk_esikleri = risk_esiklerini_olustur(youden['esik'], f2['esik'])

esikler = np.linspace(0.01, 0.99, 99)
f1_skorlari = [f1_score(y_test, (y_prob >= e).astype(int), zero_division=0) for e in esikler]
idx = int(np.argmax(f1_skorlari))
f1_optimum = {'esik': float(esikler[idx]), 'f1': float(f1_skorlari[idx])}

esik_tablosu = pd.DataFrame([
    {'yöntem': 'youden_j', 'eşik': youden['esik']},
    {'yöntem': 'f2', 'eşik': f2['esik']},
    {'yöntem': 'f1_optimum', 'eşik': f1_optimum['esik']},
])

esik_tablosu


In [ ]:
# F1 skoru ile eşik arasındaki ilişkiyi grafikte gösteriyoruz.
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(esikler, f1_skorlari, color='#4C78A8', label='F1 skoru')
ax.axvline(f1_optimum['esik'], color='#E45756', linestyle='--', label=f"F1-optimum ({f1_optimum['esik']:.2f})")
ax.set_title('Eşik - F1 İlişkisi')
ax.set_xlabel('Eşik')
ax.set_ylabel('F1')
ax.grid(alpha=0.3)
ax.legend()
plt.show()


In [ ]:
# Örnek olasılıklar üzerinden üç seviyeli risk kategorisinin nasıl davrandığını kontrol ediyoruz.
ornek_olasiliklar = [0.10, 0.25, 0.33, 0.50, 0.66, 0.80]
risk_satirlari = []
for olasilik in ornek_olasiliklar:
    kategori = risk_kategorisi_belirle(
        olasilik,
        risk_esikleri['dusuk_ust_esik'],
        risk_esikleri['orta_ust_esik'],
    )
    risk_satirlari.append({'olasilik': olasilik, 'risk_kategorisi': kategori})

pd.DataFrame(risk_satirlari)


## Kısa değerlendirme

- Kalibrasyon, modelin olasılık çıktısını yorumlamak için kritik bir adımdır.
- Eşik seçimi operasyonel hedefe göre değişebilir (recall önceliği, F1 dengesi vb.).
- Risk sınıfı tarafında uygulama kuralı sabit aralıklara göre (0.33 / 0.66) çalışır.
